# 03 — Model Training

Trains one LightGBM regressor per Big-Five trait with cross-validation.
Logs MAE and Pearson r. Saves models to `models/`.


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy import stats

from config import TRAITS, TRAIT_NAMES, LGBM_PARAMS, CV_FOLDS

## 1. Load data

In [ ]:
# df = pd.read_csv('../data/labelled_users.csv')

# Synthetic demo data
np.random.seed(42)
n = 300
feature_cols = [
    'pronoun_i_rate', 'pronoun_we_rate', 'pronoun_you_rate',
    'hedging_rate', 'certainty_rate', 'cognitive_rate', 'social_rate',
    'vader_compound_mean', 'vader_compound_std', 'vader_pos_mean', 'vader_neg_mean',
    'type_token_ratio', 'avg_sentence_length', 'flesch_kincaid_grade',
    'exclamation_rate', 'question_rate', 'caps_rate',
    'nrc_joy_rate', 'nrc_anger_rate', 'nrc_fear_rate', 'nrc_trust_rate',
    'past_tense_rate', 'present_tense_rate', 'future_tense_rate',
]

df = pd.DataFrame(np.random.rand(n, len(feature_cols)), columns=feature_cols)
for trait in TRAITS:
    df[f'{trait}_score'] = np.random.uniform(1, 7, n)

X = df[feature_cols]
y = df[[f'{t}_score' for t in TRAITS]].rename(columns={f'{t}_score': t for t in TRAITS})
print(f'X: {X.shape}  |  y: {y.shape}')

## 2. Cross-validated training

In [ ]:
results = {}
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)

for trait in TRAITS:
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   LGBMRegressor(**LGBM_PARAMS)),
    ])
    y_trait   = y[trait].values
    mae_scores = -cross_val_score(pipeline, X, y_trait, cv=kf,
                                   scoring='neg_mean_absolute_error')
    r2_scores  = cross_val_score(pipeline, X, y_trait, cv=kf, scoring='r2')
    pipeline.fit(X, y_trait)
    results[trait] = {
        'pipeline': pipeline,
        'mae': mae_scores.mean(),
        'mae_std': mae_scores.std(),
        'r2': r2_scores.mean(),
    }
    print(f'{TRAIT_NAMES[trait]:<20}  MAE={mae_scores.mean():.3f}±{mae_scores.std():.3f}  R²={r2_scores.mean():.3f}')

## 3. Feature importance

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6))
for ax, trait in zip(axes, TRAITS):
    model       = results[trait]['pipeline'].named_steps['model']
    importances = model.feature_importances_
    top_idx     = np.argsort(importances)[-10:]
    ax.barh([feature_cols[i] for i in top_idx], importances[top_idx], color='steelblue')
    ax.set_title(TRAIT_NAMES[trait], fontsize=11)
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle('Top 10 Features per Trait', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Save models

In [ ]:
import pickle
from pathlib import Path

MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

for trait, res in results.items():
    path = MODEL_DIR / f'model_{trait}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(res['pipeline'], f)
    print(f'Saved {path}')

with open(MODEL_DIR / 'feature_names.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
print('Saved feature_names.pkl')